# Train Protein-Aware E(n) Equivariant GNN on LP-PDBBind

This notebook trains the custom `GNNPredictor` with a **Unified 3D Interaction Graph** on LP-PDBBind.

**Key difference from v2**: The graph now contains **all pocket atoms** in 3D space, aligned with per-residue ESM-2 embeddings.

**Hardware:** Requires A100 GPU.

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q torch-geometric rdkit transformers accelerate

## 2. Mount Google Drive, Clone Repo & Download Data

In [ ]:
from google.colab import drive
import os, urllib.request

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/agentic-vlm'

if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/handeboyaci/agentic-vlm.git "$PROJECT_DIR"
else:
    print('Repo already cloned, pulling latest...')
    !cd "$PROJECT_DIR" && git pull

%cd "$PROJECT_DIR"

os.makedirs('data/pdbbind_deepchem/raw', exist_ok=True)
csv_path = 'data/pdbbind_deepchem/raw/LP_PDBBind.csv'
if not os.path.exists(csv_path):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/THGLab/LP-PDBBind/master/dataset/LP_PDBBind.csv',
        csv_path
    )
    print('Downloaded LP_PDBBind.csv')
else:
    print('LP_PDBBind.csv already exists.')

## 3. Dataset Preprocessing (with All-Atom 3D Graphs)

Setting `precompute_esm=True` runs ESM-2 on each protein sequence and stores
the embeddings in `data.protein_emb`. The data loader now also extracts
pocket atoms from PDB files.

**Note:** Delete your `processed/` directory if you are upgrading from sequence-only training.

In [ ]:
import torch
from torch_geometric.loader import DataLoader
from data.lp_pdbbind import LPPDBBind

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

train_full = LPPDBBind(
    root='data/pdbbind_all_atom',
    split='train',
    precompute_esm=True,
)
test_dataset = LPPDBBind(
    root='data/pdbbind_all_atom',
    split='test',
    precompute_esm=True,
)

train_full = train_full.shuffle()
val_size = int(0.1 * len(train_full))
val_dataset = train_full[:val_size]
train_dataset = train_full[val_size:]

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

sample = train_dataset[0]
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')
print(f'Nodes in sample: {sample.x.shape[0]}')
print(f'Ligand mask present: {hasattr(sample, "ligand_mask")}')

## 4. Initialize the Protein-Aware EGNN Model

In [ ]:
from models.gnn_predictor import GNNPredictor
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on {device}")

model = GNNPredictor(
    atom_feat_dim=47, 
    hidden_dim=512, 
    n_layers=6,      
    dropout=0.05,
    use_protein_encoder=True,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=8
)

def sanitize_weights(model):
    for p in model.parameters():
        if not torch.isfinite(p).all():
            p.data = torch.nan_to_num(p.data, nan=0.0)
            p.data += torch.randn_like(p.data) * 0.01

## 4.5 Resume from Checkpoint (Optional)

In [ ]:
import os, copy
CHECKPOINT_PATH = 'best_model_checkpoint.pth'
START_EPOCH = 1
best_val_loss = float('inf')
best_model_weights = None

if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint from {CHECKPOINT_PATH}...")
    checkpoint = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    START_EPOCH = checkpoint['epoch'] + 1
    best_val_loss = checkpoint['best_val_loss']
    best_model_weights = copy.deepcopy(model.state_dict())
    print(f"Resuming from epoch {START_EPOCH} (Best Val MSE: {best_val_loss:.4f})")
else:
    print("No checkpoint found. Starting from scratch.")

## 5. Training Loop (All-Atom Interaction Graph)

In [ ]:
import copy
from tqdm.auto import tqdm

def _get_graph_inputs(data):
    ligand_mask = getattr(data, 'ligand_mask', None)
    protein_res_idx = getattr(data, 'protein_res_idx', None)
    if not hasattr(data, 'protein_emb') or data.protein_emb is None: return ligand_mask, protein_res_idx, None
    try:
        batch_ids = data.batch.unique()
        embs = [data.protein_emb[b_id].to(data.x.device) for b_id in batch_ids]
        return ligand_mask, protein_res_idx, embs
    except Exception: return ligand_mask, protein_res_idx, [data.protein_emb.to(data.x.device)]

EPOCHS = 100
PATIENCE = 20
patience_counter = 0

def weighted_mse_loss(input, target):
    weights = 1.0 + (target / 5.0)
    weights = torch.clamp(weights, max=3.0)
    mse = (weights * (input - target) ** 2).mean()
    mae = (weights * torch.abs(input - target + 1e-8)).mean()
    return mse + 0.1 * mae

def compute_ranking_loss(input, target, margin=1.0):
    """Exhaustive Pairwise Ranking: Compare all pairs in the batch."""
    n = input.size(0)
    if n < 2: return torch.tensor(0.0, device=input.device)

    # Expand to all pairs (n, n)
    input_i = input.unsqueeze(1).expand(n, n)
    input_j = input.unsqueeze(0).expand(n, n)
    target_i = target.unsqueeze(1).expand(n, n)
    target_j = target.unsqueeze(0).expand(n, n)

    # Only upper triangle to avoid double counting and self-pairs
    mask = torch.triu(torch.ones(n, n, device=input.device), diagonal=1).bool()

    # Focus on pairs with a meaningful difference (> 0.5 pKa)
    diff_mask = (torch.abs(target_i - target_j) > 0.5) & mask
    if not diff_mask.any(): return torch.tensor(0.0, device=input.device)

    y = torch.sign(target_i - target_j)[diff_mask]
    
    return torch.nn.functional.margin_ranking_loss(
        input_i[diff_mask], input_j[diff_mask], y, margin=margin
    )

for epoch in range(START_EPOCH, EPOCHS + 1):
    sanitize_weights(model)
    model.train()
    train_loss, num_train_graphs, skipped = 0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch} Train')
    for data in pbar:
        data = data.to(device)
        optimizer.zero_grad()
        l_mask, p_idx, p_embs = _get_graph_inputs(data)
        out = model(data.x.float(), data.pos.float(), data.batch, ligand_mask=l_mask, protein_res_idx=p_idx, protein_embs=p_embs)
        
        # Hybrid Loss: MSE (Accuracy) + Ranking (Correlation/Decompression)
        mse_l = weighted_mse_loss(out.squeeze(), data.y.float())
        rank_l = compute_ranking_loss(out.squeeze(), data.y.float())
        loss = mse_l + 0.5 * rank_l
        
        if not torch.isfinite(loss):
            skipped += 1; continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        is_finite = all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
        if is_finite:
            optimizer.step()
            num_train_graphs += data.num_graphs
            train_loss += loss.item() * data.num_graphs
        else:
            skipped += 1; optimizer.zero_grad()
        pbar.set_postfix(skipped=skipped)
    
    train_loss = train_loss / num_train_graphs if num_train_graphs > 0 else 0.0
    model.eval()
    val_loss, num_val_graphs = 0, 0
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            l_mask, p_idx, p_embs = _get_graph_inputs(data)
            out = model(data.x.float(), data.pos.float(), data.batch, ligand_mask=l_mask, protein_res_idx=p_idx, protein_embs=p_embs)
            loss = weighted_mse_loss(out.squeeze(), data.y.float())
            if torch.isfinite(loss):
                val_loss += loss.item() * data.num_graphs
                num_val_graphs += data.num_graphs
    val_loss = val_loss / num_val_graphs if num_val_graphs > 0 else 100.0
    scheduler.step(val_loss)
    print(f'Epoch {epoch:03d} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f} | Skipped: {skipped}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model.state_dict())
        patience_counter = 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'best_val_loss': best_val_loss}, 'best_model_checkpoint.pth')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping'); break

## 6. Evaluate on Test Set

In [ ]:
import numpy as np
import scipy.stats as stats

if best_model_weights is not None:
    model.load_state_dict(best_model_weights)
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        l_mask, p_idx, p_embs = _get_graph_inputs(data)
        out = model(data.x.float(), data.pos.float(), data.batch, ligand_mask=l_mask, protein_res_idx=p_idx, protein_embs=p_embs)
        all_preds.extend(out.cpu().squeeze().tolist())
        all_targets.extend(data.y.cpu().squeeze().tolist())

preds = np.array(all_preds)
targets = np.array(all_targets)
rmse = np.sqrt(np.mean((preds - targets) ** 2))
pearson_r, _ = stats.pearsonr(preds, targets)
print(f'\nFINAL TEST SET METRICS\nRMSE: {rmse:.4f} | Pearson: {pearson_r:.4f}')

## 7. Export Trained Weights

In [ ]:
torch.save(best_model_weights, 'gnn_predictor_all_atom.pth')
print('Saved All-Atom weights to gnn_predictor_all_atom.pth')